# Employee Attrition Analysis and Prediction
**Domain:** HR Analytics

**Problem Statement:** Employee turnover poses a significant challenge for organizations. This project aims to analyze employee data, identify key drivers of attrition, and build a predictive model to support retention strategies.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Understanding
Let's load the dataset and check the first few rows.

In [ ]:
df = pd.read_csv('Employee-Attrition - Employee-Attrition (1).csv')
df.head()

In [ ]:
# Check the shape and information about the dataset
print("Shape of the dataset:", df.shape)
df.info()

## 2. Data Preprocessing & Cleaning
We will check for missing values and drop unnecessary columns like EmployeeCount, EmployeeNumber, Over18, and StandardHours because they do not provide useful information for prediction.

In [ ]:
# Checking for missing values
df.isnull().sum()

In [ ]:
# Dropping columns that have a single value or are identifiers
columns_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df.drop(columns_to_drop, axis=1, inplace=True)
df.head()

## 3. Exploratory Data Analysis (EDA)
Let's explore the distribution of our target variable `Attrition` and some other important features.

In [ ]:
# Distribution of Attrition
plt.figure(figsize=(6, 4))
sns.countplot(x='Attrition', data=df)
plt.title('Employee Attrition Count')
plt.show()

In [ ]:
# Attrition by Job Satisfaction
plt.figure(figsize=(8, 5))
sns.countplot(x='JobSatisfaction', hue='Attrition', data=df)
plt.title('Attrition by Job Satisfaction')
plt.show()

In [ ]:
# Age distribution by Attrition
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Age', hue='Attrition', kde=True)
plt.title('Age Distribution by Attrition')
plt.show()

## 4. Feature Engineering and Encoding
Machine learning models require numerical input. We will convert our categorical text data into numbers using Label Encoding.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a copy of the dataframe
df_encoded = df.copy()

le_dict = {}
for col in df_encoded.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    le_dict[col] = le

df_encoded.head()

## 5. Model Building
We will split the data into training and testing sets. Then we will train a Random Forest Classifier.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Separate features (X) and target (y)
X = df_encoded.drop('Attrition', axis=1)
y = df_encoded['Attrition']

# Split the data - 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

In [ ]:
# Initialize the model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

## 6. Model Evaluation
Let's see how well our model performs on the test data.

In [ ]:
# Make predictions
y_pred = rf_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Plotting Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 7. Feature Importance
Let's identify the key drivers of attrition based on the model.

In [ ]:
# Get feature importances
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10))
plt.title('Top 10 Key Drivers of Attrition')
plt.show()

## Conclusion
We successfully analyzed the employee data, identified key drivers of attrition (like Monthly Income, Age, and Daily Rate), and built a predictive model with good accuracy. The next step is to deploy this using Streamlit.